<a href="https://colab.research.google.com/github/SalmiFatimaZahra/Gestion-des-salaires-des-employ-s-dans-une-entreprise-/blob/main/AppliedIAFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# 1. INSTALLATION DES DEPENDANCES
# ============================================

!pip install -q ultralytics mediapipe opencv-python scikit-learn

In [ ]:
# ============================================
# 1. IMPORTS
# ============================================

import os
import json
import cv2
import math
import time
import copy
import random
import pickle
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from ultralytics import YOLO
import mediapipe as mp

warnings.filterwarnings("ignore")

In [ ]:
!pip install -q kaggle

In [ ]:
!pip install -q kaggle

from google.colab import files
import os

uploaded = files.upload()

# récupérer automatiquement le nom réel du fichier uploadé
uploaded_name = list(uploaded.keys())[0]
print("Fichier uploadé :", uploaded_name)

os.makedirs("/root/.kaggle", exist_ok=True)

# le renommer proprement en kaggle.json
with open("/root/.kaggle/kaggle.json", "wb") as f:
    f.write(uploaded[uploaded_name])

os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Configuration Kaggle terminée avec succès.")

Saving kaggle.json to kaggle (2).json
Fichier uploadé : kaggle (2).json
Configuration Kaggle terminée avec succès.


In [ ]:
# ============================================
# 2. CONFIGURATION GLOBALE
# ============================================

from pathlib import Path

PROJECT_ROOT = Path("/content/project_wlasl")
DATA_DIR = Path("/content/wlasl_data")

ANNOTATION_PATH = DATA_DIR / "nslt_100.json"
MISSING_PATH = DATA_DIR / "missing.txt"
VIDEOS_DIR = DATA_DIR / "videos"   # à vérifier si ce dossier existe bien

FEATURES_DIR = PROJECT_ROOT / "features"
MODELS_DIR = PROJECT_ROOT / "models"
LOGS_DIR = PROJECT_ROOT / "logs"

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# ----- paramètres pipeline -----
FRAME_SKIP = 2
FRAME_FEATURE_DIM = 126

# ----- paramètres modèle -----
HIDDEN_SIZE = 256
NUM_LAYERS = 2
DROPOUT = 0.5

# ----- paramètres entraînement -----
BATCH_SIZE = 16
NUM_EPOCHS = 40
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

EARLY_STOPPING_PATIENCE = 8
REDUCE_LR_PATIENCE = 3
REDUCE_LR_FACTOR = 0.5
MIN_LR = 1e-6

# ----- data augmentation -----
AUG_NOISE_STD = 0.01
AUG_SCALE_RANGE = (0.95, 1.05)
AUG_TRANSLATION_RANGE = (-0.03, 0.03)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE :", DEVICE)
print("ANNOTATION_PATH :", ANNOTATION_PATH, "| exists:", ANNOTATION_PATH.exists())
print("MISSING_PATH :", MISSING_PATH, "| exists:", MISSING_PATH.exists())
print("VIDEOS_DIR :", VIDEOS_DIR, "| exists:", VIDEOS_DIR.exists())

DEVICE : cuda
ANNOTATION_PATH : /content/wlasl_data/nslt_100.json | exists: True
MISSING_PATH : /content/wlasl_data/missing.txt | exists: True
VIDEOS_DIR : /content/wlasl_data/videos | exists: True


In [ ]:
# ============================================
# 3. REPRODUCTIBILITE
# ============================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

print(f"Seed fixée à {SEED}")

Seed fixée à 42


In [ ]:
# ============================================
# 4. CHARGEMENT DU JSON
# ============================================

import json

with open(ANNOTATION_PATH, "r", encoding="utf-8") as f:
    annotations = json.load(f)

print("Type :", type(annotations))
print("Nombre d'entrées :", len(annotations))

Type : <class 'dict'>
Nombre d'entrées : 2038


In [ ]:
# ============================================
# 5. CHARGEMENT DE missing.txt
# ============================================

def load_missing_ids(missing_path: Path):
    """
    Charge missing.txt dans un set.
    Cela permet de filtrer rapidement les vidéos signalées comme absentes.
    """
    missing_set = set()

    if not missing_path.exists():
        print(f"[INFO] missing.txt introuvable : {missing_path}")
        return missing_set

    with open(missing_path, "r", encoding="utf-8") as f:
        for line in f:
            item = line.strip()
            if item:
                missing_set.add(item)

    print("Nombre d'entrées dans missing.txt :", len(missing_set))
    return missing_set


missing_set = load_missing_ids(MISSING_PATH)

Nombre d'entrées dans missing.txt : 9103


In [ ]:
# ============================================
# 6. CHARGEMENT DU JSON
# ============================================

with open(ANNOTATION_PATH, "r", encoding="utf-8") as f:
    annotations = json.load(f)

print("Type :", type(annotations))
print("Nombre d'entrées :", len(annotations))

Type : <class 'dict'>
Nombre d'entrées : 2038


In [ ]:
# ============================================
# 6B. APERCU DE QUELQUES ENTREES
# ============================================

print("Type de annotations :", type(annotations))

if isinstance(annotations, list):
    print("Le JSON est une liste.")
    for i in range(min(2, len(annotations))):
        print(f"\n===== ENTREE {i} =====")
        print(json.dumps(annotations[i], indent=2)[:2500])

elif isinstance(annotations, dict):
    print("Le JSON est un dictionnaire.")
    print("Clés principales :", list(annotations.keys())[:20])

    for k in list(annotations.keys())[:5]:
        print(f"\n===== CLE : {k} =====")
        value = annotations[k]
        print("Type :", type(value))
        print(json.dumps(value, indent=2)[:2500] if isinstance(value, (dict, list)) else value)

else:
    print("Structure JSON non prévue :", type(annotations))

Type de annotations : <class 'dict'>
Le JSON est un dictionnaire.
Clés principales : ['05237', '69422', '10899', '10898', '10893', '10892', '10896', '10895', '10894', '51069', '51068', '51064', '51067', '51066', '51061', '51060', '66779', '66778', '65278', '43180']

===== CLE : 05237 =====
Type : <class 'dict'>
{
  "subset": "train",
  "action": [
    77,
    1,
    55
  ]
}

===== CLE : 69422 =====
Type : <class 'dict'>
{
  "subset": "val",
  "action": [
    27,
    1,
    51
  ]
}

===== CLE : 10899 =====
Type : <class 'dict'>
{
  "subset": "train",
  "action": [
    82,
    1,
    48
  ]
}

===== CLE : 10898 =====
Type : <class 'dict'>
{
  "subset": "val",
  "action": [
    82,
    1,
    39
  ]
}

===== CLE : 10893 =====
Type : <class 'dict'>
{
  "subset": "train",
  "action": [
    82,
    1,
    50
  ]
}


In [ ]:
# ============================================
# 6C. INSPECTION STRUCTURELLE
# ============================================

def inspect_json_structure(data, n_samples=3):
    print("Type global :", type(data))

    if isinstance(data, list):
        print("Longueur liste :", len(data))
        for i in range(min(n_samples, len(data))):
            item = data[i]
            print(f"\n------ Sample liste {i} ------")
            print("Type :", type(item))

            if isinstance(item, dict):
                print("Clés :", list(item.keys()))
                for k, v in item.items():
                    print(f"\n{k} -> {type(v)}")
                    if isinstance(v, list):
                        print("Longueur :", len(v))
                        if len(v) > 0:
                            first = v[0]
                            if isinstance(first, dict):
                                print(json.dumps(first, indent=2)[:1000])
                            else:
                                print(first)
                    else:
                        print(v)

    elif isinstance(data, dict):
        print("Nombre de clés :", len(data))
        keys = list(data.keys())

        for k in keys[:n_samples]:
            v = data[k]
            print(f"\n------ Clé {k} ------")
            print("Type valeur :", type(v))

            if isinstance(v, dict):
                print("Sous-clés :", list(v.keys())[:20])
                print(json.dumps(v, indent=2)[:1500])
            elif isinstance(v, list):
                print("Longueur liste :", len(v))
                if len(v) > 0:
                    first = v[0]
                    print("Type premier élément :", type(first))
                    print(json.dumps(first, indent=2)[:1500] if isinstance(first, (dict, list)) else first)
            else:
                print(v)
    else:
        print("Type non géré.")

inspect_json_structure(annotations, n_samples=3)

Type global : <class 'dict'>
Nombre de clés : 2038

------ Clé 05237 ------
Type valeur : <class 'dict'>
Sous-clés : ['subset', 'action']
{
  "subset": "train",
  "action": [
    77,
    1,
    55
  ]
}

------ Clé 69422 ------
Type valeur : <class 'dict'>
Sous-clés : ['subset', 'action']
{
  "subset": "val",
  "action": [
    27,
    1,
    51
  ]
}

------ Clé 10899 ------
Type valeur : <class 'dict'>
Sous-clés : ['subset', 'action']
{
  "subset": "train",
  "action": [
    82,
    1,
    48
  ]
}


In [ ]:
# ============================================
# 14. INSTALLATION DES DEPENDANCES VISION
# ============================================

!pip install -q ultralytics mediapipe opencv-python

In [ ]:
# ============================================
# 14B. IMPORTS COMPLEMENTAIRES
# ============================================

from ultralytics import YOLO
import mediapipe as mp

In [ ]:
# ============================================
# 15. PARAMETRES D'EXTRACTION DES FEATURES
# ============================================

FRAME_SKIP = 2                 # configurable
YOLO_MODEL_NAME = "yolov8n.pt" # modèle léger pour Colab
PERSON_CLASS_ID = 0            # classe person dans COCO
YOLO_CONF_THRESH = 0.25
CROP_MARGIN = 0.08             # marge autour de la bbox de la personne
MAX_FRAMES_DEBUG = 40          # pour les tests rapides

N_HANDS = 2
N_LANDMARKS_PER_HAND = 21
N_COORDS = 3
FRAME_FEATURE_DIM = N_HANDS * N_LANDMARKS_PER_HAND * N_COORDS  # 126

print("FRAME_FEATURE_DIM =", FRAME_FEATURE_DIM)

FRAME_FEATURE_DIM = 126


In [ ]:
# ============================================
# 16. CHARGEMENT DE YOLOv8
# ============================================

yolo_model = YOLO(YOLO_MODEL_NAME)
print("YOLO chargé :", YOLO_MODEL_NAME)

YOLO chargé : yolov8n.pt


In [ ]:
!python --version
!pip uninstall -y mediapipe protobuf
!pip install -q mediapipe==0.10.21 opencv-python ultralytics scikit-learn

Python 3.12.13
Found existing installation: mediapipe 0.10.14
Uninstalling mediapipe-0.10.14:
  Successfully uninstalled mediapipe-0.10.14
Found existing installation: protobuf 4.25.9
Uninstalling protobuf-4.25.9:
  Successfully uninstalled protobuf-4.25.9
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 11.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 req

In [ ]:
# ============================================
# 17. INITIALISATION DE MEDIAPIPE HAND LANDMARKER
# ============================================

!pip install -q mediapipe

import os
import urllib.request
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Modèle officiel Hand Landmarker
HAND_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
HAND_MODEL_PATH = "/content/hand_landmarker.task"

if not os.path.exists(HAND_MODEL_PATH):
    urllib.request.urlretrieve(HAND_MODEL_URL, HAND_MODEL_PATH)

BaseOptions = python.BaseOptions
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions
VisionRunningMode = vision.RunningMode

hand_options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=HAND_MODEL_PATH),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

hands_detector = HandLandmarker.create_from_options(hand_options)

print("MediaPipe Hand Landmarker prêt.")
print("Modèle :", HAND_MODEL_PATH)

ModuleNotFoundError: No module named 'mediapipe.tasks.c'

In [ ]:
# ============================================
# 17. VERIFICATION DE MEDIAPIPE
# ============================================

import mediapipe as mp

print("Module mediapipe :", mp)
print("Type :", type(mp))
print("Has solutions :", hasattr(mp, "solutions"))

if hasattr(mp, "__file__"):
    print("Chemin du module :", mp.__file__)

Module mediapipe : <module 'mediapipe' from '/usr/local/lib/python3.12/dist-packages/mediapipe/__init__.py'>
Type : <class 'module'>
Has solutions : False
Chemin du module : /usr/local/lib/python3.12/dist-packages/mediapipe/__init__.py


In [ ]:
import cv2
import numpy as np
import mediapipe as mp

print("mediapipe version :", getattr(mp, "__version__", "inconnue"))
print("has solutions :", hasattr(mp, "solutions"))
print("module path :", getattr(mp, "__file__", "inconnu"))

mediapipe version : 0.10.33
has solutions : False
module path : /usr/local/lib/python3.12/dist-packages/mediapipe/__init__.py


In [ ]:
# ============================================
# 17B. REINSTALLATION PROPRE DE MEDIAPIPE
# ============================================

!pip uninstall -y mediapipe
!pip install -q mediapipe==0.10.14

Found existing installation: mediapipe 0.10.33
Uninstalling mediapipe-0.10.33:
  Successfully uninstalled mediapipe-0.10.33
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 33.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.


In [ ]:
# ============================================
# 17C. REIMPORT APRES RESTART
# ============================================

import mediapipe as mp

print("Module mediapipe :", mp)
print("Has solutions :", hasattr(mp, "solutions"))
print("Version :", getattr(mp, "__version__", "version inconnue"))

Module mediapipe : <module 'mediapipe' from '/usr/local/lib/python3.12/dist-packages/mediapipe/__init__.py'>
Has solutions : False
Version : 0.10.33
